# ПР6. Классификация текстов: корпус статей Википедии

**Задача:** по тексту статьи определить рубрику — «Астероиды» или «Планеты».

**Процедура разметки (выполнена автоматически):** корпус собран из
русскоязычной Википедии через официальный API. Метка класса = рубрика
статьи (категория Википедии), к которой относится статья. Объективная
разметка, ошибки человека-разметчика исключены. Объём корпуса > 500
документов (уровень C).

**Пайплайн:** очистка текста → векторизация (BoW / TF-IDF) →
классификатор → анализ ошибок.

**Важно:** векторизатор обучается только на train (классическая утечка
текстовых задач).

In [1]:
import requests
import time
import random
import os
import pandas as pd

CSV_PATH = "../data/processed/corpus_wiki.csv"

if os.path.exists(CSV_PATH):
    print("Корпус уже собран, сбор заголовков пропущен")
else:
    API = "https://ru.wikipedia.org/w/api.php"
    HEADERS = {
        "User-Agent": "AsteroidMLStudentProject/1.0 (educational; contact: student@example.com)"
    }

    def get_category_content(category, limit=400, depth=2):
        """Статьи из категории + рекурсия в подкатегории (глубина depth)."""
        pages = []
        queue = [(category, depth)]
        visited = set()

        while queue and len(pages) < limit:
            cat, d = queue.pop(0)
            if cat in visited:
                continue
            visited.add(cat)

            params = {
                "action": "query", "list": "categorymembers",
                "cmtitle": f"Категория:{cat}",
                "cmlimit": "500", "format": "json"
            }
            r = requests.get(API, params=params, headers=HEADERS, timeout=30)
            if r.status_code != 200:
                break
            members = r.json()["query"]["categorymembers"]

            for m in members:
                if m["ns"] == 0:  # ns=0 — обычные статьи
                    pages.append(m["title"])
                elif m["ns"] == 14 and d > 0:  # ns=14 — подкатегории
                    sub = m["title"].replace("Категория:", "")
                    queue.append((sub, d - 1))

            time.sleep(0.3)

        return list(dict.fromkeys(pages))[:limit]  # уникальные, сохраняя порядок

    asteroid_titles = get_category_content("Астероиды", limit=350, depth=2)
    print("Астероиды:", len(asteroid_titles))

    planet_titles = get_category_content("Планеты Солнечной системы", limit=350, depth=2)
    print("Планеты:", len(planet_titles))

    print("\nПримеры (астероиды):", asteroid_titles[:5])
    print("Примеры (планеты):", planet_titles[:5])

Корпус уже собран, сбор заголовков пропущен


In [2]:
import pandas as pd
import os

CSV_PATH = "../data/processed/corpus_wiki.csv"

corpus = pd.read_csv(CSV_PATH)
if "title" not in corpus.columns:
    corpus["title"] = ""   # старый формат — просто добавляем колонку

print("Итого документов:", len(corpus))
print(corpus["label"].value_counts())
print("\nСредняя длина текста, символов:", int(corpus["text"].str.len().mean()))
corpus.head(3)

Итого документов: 560
label
planet      312
asteroid    248
Name: count, dtype: int64

Средняя длина текста, символов: 4980


,text,label,title
0,Астеро́ид (распространённый до 2006 года синон...,asteroid,
1,Покрытие звёзд астероидом — это астрономическо...,asteroid,
2,1999 XS35 — астероид — аполлон. Его особенност...,asteroid,


## Очистка и разбиение корпуса

Очистка: lowercase, удаление пунктуации и цифр, удаление стоп-слов
(стандартный список русских стоп-слов scikit-learn).

Разбиение train/test выполняется **до векторизации** — словарь
векторизатора обучается только на train, иначе утечка данных,
специфичная для текстовых задач.

In [3]:
import re
from sklearn.model_selection import train_test_split

def clean_text(text):
    """Очистка: lowercase, без пунктуации/цифр/стоп-слов."""
    text = text.lower()
    text = re.sub(r"[^а-яa-zё\s]", " ", text)  # оставляем только буквы
    text = re.sub(r"\s+", " ", text).strip()
    return text

corpus["text_clean"] = corpus["text"].apply(clean_text)

X_text = corpus["text_clean"]
y_text = corpus["label"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X_text, y_text, test_size=0.25, random_state=42, stratify=y_text
)

print("Train:", len(X_tr), "| Test:", len(X_te))
print(y_tr.value_counts())
print("\nПример после очистки:")
print(X_tr.iloc[0][:300])

Train: 420 | Test: 140
label
planet      234
asteroid    186
Name: count, dtype: int64

Пример после очистки:
внутренний спутник англ inner moon естественный спутник с проградным движением на орбите с малым наклоном внутри области крупных спутников планеты обычно предполагается что такие спутники сформировались в том же месте и в то же время что и планета исключением являются естественные спутники нептуна п


In [4]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score)

# Русские стоп-слова (scikit-learn в новых версиях не содержит встроенный
# список для русского — используем собственный компактный список)
RUSSIAN_STOP_WORDS = [
    "и", "в", "на", "с", "по", "за", "из", "у", "о", "от", "до", "не", "но",
    "что", "как", "а", "это", "его", "ее", "их", "к", "для", "или", "он",
    "она", "они", "мы", "вы", "я", "ты", "быть", "был", "была", "были",
    "есть", "the", "of", "is", "was", "and", "in", "on", "to", "at",
    "более", "менее", "также", "который", "которая", "которые", "при",
    "так", "все", "уже", "еще", "может", "только", "даже", "где", "когда",
    "однако", "между", "чем", "после", "перед", "над", "под", "без",
]

def evaluate_vectorizer(vec, name):
    """Обучить векторизатор только на train, оценить на test."""
    X_tr_vec = vec.fit_transform(X_tr)   # fit только на train!
    X_te_vec = vec.transform(X_te)       # test только трансформируем

    model = MultinomialNB()
    model.fit(X_tr_vec, y_tr)
    y_pred = model.predict(X_te_vec)

    print(f"\n=== {name} ===")
    print(f"Размерность признакового пространства: {X_tr_vec.shape[1]}")
    print(f"Accuracy:  {accuracy_score(y_te, y_pred):.4f}")
    print(f"Precision: {precision_score(y_te, y_pred, pos_label='asteroid'):.4f}")
    print(f"Recall:    {recall_score(y_te, y_pred, pos_label='asteroid'):.4f}")
    print(f"F1:        {f1_score(y_te, y_pred, pos_label='asteroid'):.4f}")

    return model, vec, y_pred

# Вариант 1: Bag of Words (частоты слов)
model_bow, vec_bow, y_pred_bow = evaluate_vectorizer(
    CountVectorizer(stop_words=RUSSIAN_STOP_WORDS, min_df=2),
    "CountVectorizer (BoW)"
)

# Вариант 2: TF-IDF
model_tfidf, vec_tfidf, y_pred_tfidf = evaluate_vectorizer(
    TfidfVectorizer(stop_words=RUSSIAN_STOP_WORDS, min_df=2),
    "TfidfVectorizer"
)


=== CountVectorizer (BoW) ===
Размерность признакового пространства: 14761
Accuracy:  0.9286
Precision: 0.8824
Recall:    0.9677
F1:        0.9231

=== TfidfVectorizer ===
Размерность признакового пространства: 14761
Accuracy:  0.9571
Precision: 0.9516
Recall:    0.9516
F1:        0.9516


### Сравнение BoW и TF-IDF

| Метрика | BoW | TF-IDF |
|---|---|---|
| Accuracy | 0.9286 | **0.9571** |
| Precision | 0.8824 | **0.9516** |
| Recall | 0.9677 | 0.9516 |
| F1 | 0.9231 | **0.9516** |

Оба векторизатора построили словарь из 14 761 слова (min_df=2,
стоп-слова удалены). TF-IDF превзошёл BoW по accuracy и F1, при этом
сбалансировав precision и recall (0.95/0.95 против 0.88/0.97).

**Интерпретация:** тексты двух рубрик близки по тематике (космос) и
содержат много общих слов («орбита», «солнечная система», «открыт»).
BoW сильнее реагирует на такие частотные общие слова, тогда как TF-IDF
понижает их вес и выделяет характерные для класса термины. Для данного
корпуса TF-IDF — предпочтительный метод векторизации.

In [5]:
# Берём лучшую модель (TF-IDF) и ищем ошибки на test
misclassified = []

for idx, (true, pred) in enumerate(zip(y_te, y_pred_tfidf)):
    if true != pred:
        misclassified.append({
            "текст": X_te.iloc[idx][:250],  # начало очищенного текста
            "истина": true,
            "предсказание": pred
        })

errors_df = pd.DataFrame(misclassified)
print("Всего ошибок:", len(errors_df))
errors_df.head(5)

Всего ошибок: 6


,текст,истина,предсказание
0,косми ческое выве тривание обобщающее название...,asteroid,planet
1,дактиль спутник астероида ида официальное обоз...,asteroid,planet
2,подковообразная орбита это один из типов коорб...,asteroid,planet
3,хуфу араб небольшой околоземный астероид из гр...,planet,asteroid
4,захват астероида англ asteroid capture переход...,planet,asteroid


### Анализ ошибок модели (TF-IDF)

Всего ошибок на test (140 документов): ~6. Разбор 5 случаев:

1. **«Космическое выветривание»** (asteroid→planet): общенаучная статья
   без специфических терминов ни одного класса; решение принято
   по общим «космическим» словам. *Гипотеза: нехватка характерных признаков.*
2. **«Дакти́ль»** (asteroid→planet): статья про астероид, но слово
   «спутник» — сильный маркер класса planet. *Гипотеза: доминирование
   одного ключевого слова над общим смыслом.*
3. **«Подковообразная орбита»** (asteroid→planet): чистая небесная
   механика, классовая специфика отсутствует. *Гипотеза: как и №1.*
4. **«Хуфу»** (planet→asteroid, модель ПРАВА): текст явно описывает
   астероид, но эталонная метка — planet. Это **шум автоматической
   разметки**: статья числится в подкатегории рубрики «Планеты».
   *Вывод: часть ошибок — не недостаток модели, а недостаток разметки
   корпуса.* Для продакшн-системы такие случаи нужно чистить вручную.
5. **«Захват астероида»** (planet→asteroid): текст описывает процесс
   с частыми упоминаниями планет. *Гипотеза: перекос частот по
   ассоциированным объектам, а не по теме статьи.*

**Общий вывод:** ошибки сосредоточены на обще-научных и процессуальных
статьях, а не на статьях-объектах. Модель уверенно различает классы
по характерной лексике; её слабое место — тексты на стыке тем.